# Exercitiul 2 - Extragere informatii vizuale: Jocul X si 0

**Varianta:** Imagini cu tabla de X si 0 (3x3). Trebuie sa detectezi:
- **Task 1:** Numarul de celule ocupate + pozitia lor (1-9, de sus-jos, stanga-dreapta)
- **Task 2:** Simbolul din fiecare celula ocupata (X sau 0)

**Format output** (fisier `.txt` cu acelasi nume ca imaginea):
```
nr_celule_ocupate
pozitie simbol
pozitie simbol
...
```
Exemplu:
```
4
1 0
2 X
3 X
5 0
```

**Scor:** 0.125 per imagine corecta completa (Task1 + Task2)

In [2]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

In [3]:
# ============================================================
# CONFIGURARE
# ============================================================

TRAIN_DIR  = "data/ex2/antrenare"
TEST_DIR   = "data/ex2/testare"
OUTPUT_DIR = "data/ex2/output"

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [4]:
# ============================================================
# PASUL 1: DETECTARE TABLA (contur dreptunghiular mare)
# Ideea: gasim cel mai mare contur dreptunghiular din imagine
# si facem o transformare de perspectiva pentru a-l indrepta.
# ============================================================

def detecteaza_tabla(img):
    """
    Returneaza imaginea tablei de joc, indrepdata (bird's eye view).
    Daca nu gaseste tabla, returneaza imaginea originala.
    """
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (5, 5), 0)
    edges = cv2.Canny(blur, 50, 150)

    contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    contours = sorted(contours, key=cv2.contourArea, reverse=True)

    for cnt in contours[:5]:
        perimeter = cv2.arcLength(cnt, True)
        approx = cv2.approxPolyDP(cnt, 0.02 * perimeter, True)
        if len(approx) == 4:  # patrulater/dreptunghi
            pts = approx.reshape(4, 2).astype(np.float32)
            # Sorteaza punctele: top-left, top-right, bottom-right, bottom-left
            s = pts.sum(axis=1)
            diff = np.diff(pts, axis=1)
            ordered = np.array([
                pts[np.argmin(s)],    # top-left
                pts[np.argmin(diff)], # top-right
                pts[np.argmax(s)],    # bottom-right
                pts[np.argmax(diff)]  # bottom-left
            ], dtype=np.float32)

            SIZE = 450  # dimensiunea imaginii de output
            dst = np.array([[0,0],[SIZE,0],[SIZE,SIZE],[0,SIZE]], dtype=np.float32)
            M = cv2.getPerspectiveTransform(ordered, dst)
            warped = cv2.warpPerspective(img, M, (SIZE, SIZE))
            return warped, True

    # Fallback: returneaza imaginea originala redimensionata
    return cv2.resize(img, (450, 450)), False

# Test pe o imagine de antrenare
img_files = sorted([f for f in os.listdir(TRAIN_DIR) if f.endswith('.jpg') or f.endswith('.png')])
if img_files:
    sample = cv2.imread(os.path.join(TRAIN_DIR, img_files[0]))
    warped, ok = detecteaza_tabla(sample)
    plt.figure(figsize=(10,5))
    plt.subplot(1,2,1); plt.imshow(cv2.cvtColor(sample, cv2.COLOR_BGR2RGB)); plt.title('Original')
    plt.subplot(1,2,2); plt.imshow(cv2.cvtColor(warped, cv2.COLOR_BGR2RGB)); plt.title(f'Tabla detectata: {ok}')
    plt.show()

FileNotFoundError: [WinError 3] The system cannot find the path specified: 'data/ex2/antrenare'

In [ ]:
# ============================================================
# PASUL 2: IMPARTIRE IN 9 CELULE
# Tabla e 3x3, deci o impartim in 9 celule egale.
# Celulele sunt numerotate 1-9, de sus in jos, stanga la dreapta.
# ============================================================

def get_celule(tabla_img):
    """
    Primeste imaginea tablei (450x450) si returneaza
    un dict {pozitie: imagine_celula} unde pozitie e 1-9.
    """
    h, w = tabla_img.shape[:2]
    cell_h = h // 3
    cell_w = w // 3
    celule = {}
    pos = 1
    for row in range(3):
        for col in range(3):
            y1 = row * cell_h
            y2 = (row + 1) * cell_h
            x1 = col * cell_w
            x2 = (col + 1) * cell_w
            # Margine mica pentru a evita liniile grilei
            margin = 8
            celule[pos] = tabla_img[y1+margin:y2-margin, x1+margin:x2-margin]
            pos += 1
    return celule

# Vizualizeaza celulele
if img_files:
    celule = get_celule(warped)
    fig, axes = plt.subplots(3, 3, figsize=(8, 8))
    for i in range(1, 10):
        ax = axes[(i-1)//3][(i-1)%3]
        ax.imshow(cv2.cvtColor(celule[i], cv2.COLOR_BGR2RGB))
        ax.set_title(f'Celula {i}')
        ax.axis('off')
    plt.tight_layout()
    plt.show()

In [ ]:
# ============================================================
# PASUL 3: DETECTARE CELULA GOALA / OCUPATA
# Idee: o celula goala are culoarea tablei (beige/maro).
# O celula ocupata are pixeli albi (X) sau colorati (0).
# Pragul se seteaza pe baza imaginilor de antrenare.
# ============================================================

def este_ocupata(cell_img, prag_alb=50):
    """
    Returneaza True daca celula contine un simbol.
    Metoda: comparam numarul de pixeli albi (> prag) sau
    varianta colorata cu media celulei.
    Ajusteaza prag_alb dupa inspectia datelor de antrenare!
    """
    gray = cv2.cvtColor(cell_img, cv2.COLOR_BGR2GRAY)
    # Binarizare Otsu
    _, thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    # Numarul de pixeli albi
    nr_albi = np.sum(thresh == 255)
    total   = thresh.size
    procent = nr_albi / total
    return procent > prag_alb / 100.0  # <-- CALIBREAZA prag_alb

# Calibrare pe antrenare - vizualizeaza procentul de alb per celula
if img_files:
    for i in range(1, 10):
        gray = cv2.cvtColor(celule[i], cv2.COLOR_BGR2GRAY)
        _, thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        procent = np.sum(thresh == 255) / thresh.size * 100
        print(f"Celula {i}: {procent:.1f}% alb")

In [ ]:
# ============================================================
# PASUL 4: CLASIFICARE SIMBOL (X sau 0)
# Idee:
# - 0 (cercul) are contur circular -> folosim HoughCircles SAU
#   detectam un contur inchis mare fara colturi ascutite
# - X are doua linii diagonale -> folosim contururile sau
#   forma bounding box vs aria
# Alternativ: comparam cu template-uri din antrenare.
# ============================================================

def clasifica_simbol(cell_img):
    """
    Returneaza 'X' sau '0'.
    Metoda simpla: detectam cercluri cu HoughCircles.
    Daca gasim un cerc -> '0', altfel -> 'X'.
    """
    gray = cv2.cvtColor(cell_img, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (5, 5), 0)

    # Incearca detectie cerc
    circles = cv2.HoughCircles(
        blur, cv2.HOUGH_GRADIENT, dp=1,
        minDist=20,
        param1=50, param2=20,         # <-- CALIBREAZA
        minRadius=10, maxRadius=80    # <-- CALIBREAZA
    )

    if circles is not None:
        return '0'
    return 'X'

# --- ALTERNATIVA prin contururi (mai robusta uneori) ---
def clasifica_simbol_v2(cell_img):
    """
    Alternativa: folosim circularitatea conturului principal.
    circularitate = 4*pi*arie / perimetru^2
    -> aproape de 1 = cerc (0), departe de 1 = X
    """
    gray = cv2.cvtColor(cell_img, cv2.COLOR_BGR2GRAY)
    _, thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if not contours:
        return 'X'  # fallback

    # Cel mai mare contur
    cnt = max(contours, key=cv2.contourArea)
    area = cv2.contourArea(cnt)
    perim = cv2.arcLength(cnt, True)
    if perim == 0:
        return 'X'

    circularitate = 4 * np.pi * area / (perim ** 2)
    # print(f"Circularitate: {circularitate:.3f}")
    return '0' if circularitate > 0.6 else 'X'   # <-- CALIBREAZA pragul

In [ ]:
# ============================================================
# PASUL 5: CALIBRARE PE ANTRENARE
# Citeste adnotarile si verifica acuratetea metodelor de mai sus.
# ============================================================

def citeste_adnotare(txt_path):
    """
    Citeste fisierul de adnotare.
    Format:
    nr_celule
    pozitie simbol
    ...
    Returneaza dict {pozitie: simbol}
    """
    result = {}
    with open(txt_path, 'r') as f:
        lines = f.read().strip().split('\n')
    nr = int(lines[0].strip())
    for line in lines[1:nr+1]:
        parts = line.strip().split()
        if len(parts) >= 2:
            result[int(parts[0])] = parts[1]
    return result

# Evalueaza pe antrenare
correct_task1 = 0
correct_task2 = 0
total = 0

# Ajusteaza pragul de "ocupata" global
PRAG_OCUPATA = 50  # <-- CALIBREAZA

for fname in sorted(os.listdir(TRAIN_DIR)):
    if not (fname.endswith('.jpg') or fname.endswith('.png')):
        continue
    txt_name = os.path.splitext(fname)[0] + '.txt'
    txt_path = os.path.join(TRAIN_DIR, txt_name)
    if not os.path.exists(txt_path):
        continue

    img = cv2.imread(os.path.join(TRAIN_DIR, fname))
    tabla, _ = detecteaza_tabla(img)
    celule   = get_celule(tabla)
    gt       = citeste_adnotare(txt_path)

    # Predictie
    pred = {}
    for pos in range(1, 10):
        if este_ocupata(celule[pos], prag_alb=PRAG_OCUPATA):
            simbol = clasifica_simbol_v2(celule[pos])
            pred[pos] = simbol

    # Task 1: pozitii corecte?
    t1_ok = set(pred.keys()) == set(gt.keys())
    # Task 2: simboluri corecte? (doar daca Task1 e corect)
    t2_ok = t1_ok and all(pred.get(p) == gt[p] for p in gt)

    correct_task1 += int(t1_ok)
    correct_task2 += int(t2_ok)
    total += 1

print(f"Task1 (pozitii): {correct_task1}/{total} = {correct_task1/total*100:.1f}%")
print(f"Task2 (simboluri): {correct_task2}/{total} = {correct_task2/total*100:.1f}%")

In [ ]:
# ============================================================
# PASUL 6: GENERARE OUTPUT PENTRU TESTARE
# ============================================================

for fname in sorted(os.listdir(TEST_DIR)):
    if not (fname.endswith('.jpg') or fname.endswith('.png')):
        continue

    img   = cv2.imread(os.path.join(TEST_DIR, fname))
    tabla, _ = detecteaza_tabla(img)
    celule   = get_celule(tabla)

    ocupate = {}
    for pos in range(1, 10):
        if este_ocupata(celule[pos], prag_alb=PRAG_OCUPATA):
            ocupate[pos] = clasifica_simbol_v2(celule[pos])

    # Scrie fisierul de output
    out_name = os.path.splitext(fname)[0] + '.txt'
    out_path = os.path.join(OUTPUT_DIR, out_name)
    with open(out_path, 'w') as f:
        f.write(f"{len(ocupate)}\n")
        for pos in sorted(ocupate.keys()):
            f.write(f"{pos} {ocupate[pos]}\n")

print(f"Output generat in: {OUTPUT_DIR}")
print("Exemplu output:")
first_out = sorted(os.listdir(OUTPUT_DIR))[0]
print(open(os.path.join(OUTPUT_DIR, first_out)).read())